# 05-01 데이터 준비와 라벨 만들기

런타임 유형을 GPU로 바꾼 뒤 첫 셀부터 차례로 실행하세요.

In [ ]:
import json
import os
import random
import re
import urllib.request

import torch
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

## 1. 문장 모으기

[KLUE](https://github.com/KLUE-benchmark/KLUE) NLI 학습 데이터의 문장을 씁니다. (CC BY-SA 4.0)

In [ ]:
URL = "https://raw.githubusercontent.com/KLUE-benchmark/KLUE/main/klue_benchmark/klue-nli-v1.1/klue-nli-v1.1_train.json"
if not os.path.exists("klue_nli_train.json"):
    urllib.request.urlretrieve(URL, "klue_nli_train.json")

with open("klue_nli_train.json", encoding="utf-8") as f:
    data = json.load(f)
raw = sorted({s for d in data for s in (d["premise"], d["hypothesis"])})
print(len(raw))
print(raw[100:103])

## 2. 정리하기

In [ ]:
def clean(s):
    s = re.sub(r"\s+", " ", s).strip()  # 연속된 공백을 하나로
    return s

sentences = [clean(s) for s in raw]
sentences = [s for s in sentences if 10 <= len(s) <= 120 and " " in s]
random.seed(0)
random.shuffle(sentences)
print(len(sentences))

In [ ]:
lengths = [len(s) for s in sentences]
plt.figure(figsize=(6, 3))
plt.hist(lengths, bins=50, color="tab:blue")
plt.xlabel("characters per sentence")
plt.ylabel("count")
plt.show()
print("평균 길이:", sum(lengths) / len(lengths))

## 3. 입력과 라벨 만들기

In [ ]:
def make_example(sentence):
    chars, labels = [], []
    for i, ch in enumerate(sentence):
        if ch == " ":
            continue
        chars.append(ch)
        next_is_space = i + 1 < len(sentence) and sentence[i + 1] == " "
        labels.append(1 if next_is_space else 0)
    return "".join(chars), labels

text, labels = make_example("나는 학교에 간다")
print(text)
print(labels)
for ch, y in zip(text, labels):
    print(ch, y)

In [ ]:
def apply_spacing(text, labels):
    out = []
    for ch, y in zip(text, labels):
        out.append(ch)
        if y == 1:
            out.append(" ")
    return "".join(out).strip()

print(apply_spacing(text, labels))
print(all(apply_spacing(*make_example(s)) == s for s in sentences))

In [ ]:
all_labels = [y for s in sentences for y in make_example(s)[1]]
print(f"라벨 1의 비율: {sum(all_labels) / len(all_labels):.3f}")

## 4. 학습, 검증, 테스트로 나누기

In [ ]:
n = len(sentences)
train_s = sentences[: int(n * 0.9)]
val_s = sentences[int(n * 0.9) : int(n * 0.95)]
test_s = sentences[int(n * 0.95) :]
print(len(train_s), len(val_s), len(test_s))

## 5. 토크나이저 (03-01의 CharTokenizer)

In [ ]:
class CharTokenizer:
    def __init__(self, texts, special=("<pad>", "<unk>")):
        chars = sorted(set("".join(texts)))
        self.itos = list(special) + chars
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.pad_id = self.stoi["<pad>"]
        self.unk_id = self.stoi["<unk>"]

    def __len__(self):
        return len(self.itos)

    def encode(self, text):
        return [self.stoi.get(ch, self.unk_id) for ch in text]

    def decode(self, ids):
        return "".join(self.itos[i] for i in ids if i != self.pad_id)

tok = CharTokenizer([make_example(s)[0] for s in train_s])  # 학습 데이터로만 만들어요
print("어휘집 크기:", len(tok))

## 6. Dataset과 DataLoader

In [ ]:
class SpacingDataset(Dataset):
    def __init__(self, sentences, tok):
        self.items = []
        for s in sentences:
            text, labels = make_example(s)
            self.items.append((tok.encode(text), labels))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]

def collate(batch, pad_id=0):
    max_len = max(len(ids) for ids, _ in batch)
    ids = torch.full((len(batch), max_len), pad_id)  # (B, T)
    labels = torch.full((len(batch), max_len), -100)  # (B, T), -100은 손실에서 빼요
    for i, (x, y) in enumerate(batch):
        ids[i, : len(x)] = torch.tensor(x)
        labels[i, : len(y)] = torch.tensor(y)
    return ids, labels

train_ds = SpacingDataset(train_s, tok)
train_dl = DataLoader(train_ds, batch_size=4, shuffle=True, collate_fn=collate)
ids, labels = next(iter(train_dl))
print(ids.shape, labels.shape)
print(labels[0])
print(tok.decode(ids[0].tolist()))

In [ ]:
plt.figure(figsize=(12, 2))
plt.imshow(labels.clamp(min=-1), cmap="coolwarm", vmin=-1, vmax=1)
plt.title("labels: red = space after, blue-gray = no space, dark blue = padding (-100)")
plt.xlabel("position")
plt.yticks(range(labels.shape[0]))
plt.show()